## Solucion: 03_pandas_pipeline
### Ing. Jorge Uriel liévano Cifuentes
### viernes 19 de junio de 2026
### Bucaramanga - Colombia - UDES

In [1]:
# PASO 1 — Detectar automáticamente la codificación
# pip install chardet

import chardet

# Detectar codificación del archivo
with open('../data/ventas_techventas.csv', 'rb') as f:
    encoding = chardet.detect(f.read())['encoding']
    print(f"✅ Codificación detectada: {encoding}")

print(encoding)

✅ Codificación detectada: iso8859-16
iso8859-16


## * Ingeniería de Datos con Pandas *
#### Carga, limpia, transforma y exporta el dataset completo

## Paso 1 - Configurar el path para importar librerías personalizadas desde la carpeta scripts/
### Este notebook y como parte de la solución del Pipeline se implementó una librería 'limpiar_csv_completo' para facilitar la limpieza del DataFrame que contiene el archivo suministrado 'ventas_techventas.csv'

In [2]:
import sys
import csv

sys.path.append("..")

# Importa la librería que permite realizar la limpieza del dataset
from scripts.limpiar_csv_completo import limpiar_csv_completo

print("✅ Importación exitosa")

✅ Importación exitosa


## PASO 2 - Cargar correctamente el DataFrame

In [3]:
# Carga de datos desde CSV
# CSV: Es texto plano donde las columnas está separadas por un delimitador.
# sep = Definir el caracter separador de columnas
# encoding = Define com estan codificados los caracteres:
# utf-8
# latin1
# cp1252
import pandas as pd

df = pd.read_csv(
    '../data/ventas_techventas.csv',
    sep=',',
    quotechar='"',    # Carácter que delimita campos de texto
    quoting=csv.QUOTE_NONE, # Sin comillas en ningún campo
    escapechar='\\',
    doublequote=True, # Maneja comillas dobles dentro del texto    
    encoding='latin1',
    parse_dates=['fecha'],
    engine='python',  # Más lento pero más tolerante
    # Esta columna la interprete como fecha.
)

# Verificar que la tupla 2 se cargó correctamente
print(df.iloc[2])  # Tupla 2 (índice 1)
print(f"Producto: {df.iloc[1]['producto']}")  # Debe mostrar: Monitor 27"

print(f"✅ Datos cargados: {len(df)} registros")
print(f"📊 Columnas: {df.columns.tolist()}")

order_id                         "1003
fecha              2024-01-10 00:00:00
cliente_id                        C003
cliente_nombre           DataSolutions
region                          Centro
producto                  Monitor 27""
categoria                  Electronica
cantidad                             5
precio_unitario                  350.0
descuento                         0.05
vendedor                   Ana Lï¿½ï¿"
Name: 2, dtype: object
Producto: Mouse Inalambrico
✅ Datos cargados: 60 registros
📊 Columnas: ['order_id', 'fecha', 'cliente_id', 'cliente_nombre', 'region', 'producto', 'categoria', 'cantidad', 'precio_unitario', 'descuento', 'vendedor']


 ## PASO 3 — Verificar estructura del DataFrame que contiene la estructura del archivo 'ventas_techventas.csv'

In [4]:
# 3.1 Verificar los nombres de las columnas y los tipos de valores
df.head()

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
0,1001,2024-01-05,C001,TechCorp SA,Norte,Laptop Pro 15,Electronica,2,1200.0,0.10,Ana Lï¿½ï¿
1,1002,2024-01-07,C002,Innovatek,Sur,Mouse Inalambrico,Perifericos,15,25.0,0.00,Carlos Ruiz
2,"""1003",2024-01-10,C003,DataSolutions,Centro,"Monitor 27""""",Electronica,5,350.0,0.05,"Ana Lï¿½ï¿"""
3,1004,2024-01-12,C001,TechCorp SA,Norte,Teclado Mecanico,Perifericos,10,85.0,0.10,Luis Mora
4,1005,2024-01-15,C004,CloudNet,Este,Laptop Pro 15,Electronica,3,1200.0,0.15,Carlos Ruiz


In [5]:
# 3.2 Estructura de los datos, mostrando el tipo de dato de cada columna, la cantidad de valores no nulos y el consumo total de memoria
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         60 non-null     str           
 1   fecha            60 non-null     datetime64[us]
 2   cliente_id       60 non-null     str           
 3   cliente_nombre   60 non-null     str           
 4   region           60 non-null     str           
 5   producto         59 non-null     str           
 6   categoria        60 non-null     str           
 7   cantidad         60 non-null     int64         
 8   precio_unitario  60 non-null     float64       
 9   descuento        60 non-null     float64       
 10  vendedor         60 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(1), str(7)
memory usage: 5.3 KB


In [6]:
# 3.3 Determinar tamaño del DataFrame, número de filas y número de columnas
df.shape

(60, 11)

## RESUMEN ANOMALÍAS OBSERVADAS
### El dataset contiene:
### • 60 registros y 11 columnas
### • 11 registros corruptos por formato CSV
### • Valores nulo en 11 registros por cada columna partir de la columna 'fecha' en adelante
### • 1 cantidad negativa
### • 1 posible outlier
### • Productos vacíos
### • Errores de codificación de texto
### • Según documento los tipos de datos para 'order_id' y 'cantidad' deben ser ENTERO y según lo detectado son CADENA
### Será necesario realizar un proceso de limpieza antes de efectuar análisis de negocio.

## PASO 4 - Identificar nulos

In [7]:
"""
Cuenta la cantidad de valores nulos o faltantes (NaN) en cada columna de un DataFrame
"""
df.isna().sum()

order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           1
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64

In [8]:
"""
Filtra y devuelve todas las filas del DataFrame que contienen al menos un valor nulo (como NaN o None)
"""
df[df.isna().any(axis=1)]


,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2,350.0,0.05,Maria Torres


## PASO 5 - Identificar productos vacíos

In [9]:
"""
Devuelve solo las filas donde la columna 'producto' tiene valores nulos o faltantes (como NaN o None)
"""
df[df['producto'].isna()]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
42,1043,2024-05-01,C010,DeltaOps,Norte,NaN,Electronica,2,350.0,0.05,Maria Torres


## PASO 6 - Identificar cantidades negativas

In [10]:
df[df['cantidad'] < 0]

,order_id,fecha,cliente_id,cliente_nombre,region,producto,categoria,cantidad,precio_unitario,descuento,vendedor
47,1048,2024-05-15,C007,AlphaTech,Sur,Laptop Pro 15,Electronica,-1,1200.0,0.0,Ana Lï¿½ï¿


## PASO 7 - Ejecutar limpieza

In [11]:
ruta_entrada = '../data/ventas_techventas.csv'
ruta_salida = '../data/ventas_techventas_clean.csv'

df_limpio = limpiar_csv_completo(ruta_entrada, ruta_salida)

🚀 LIMPIEZA COMPLETA DE CSV

📂 Paso 1: Leyendo y reparando encoding...
🔧 Paso 2: Reparando encoding...
🔧 Paso 3: Reparando comillas mal formadas...
💾 Paso 4: Guardando archivo temporal...
📊 Paso 5: Cargando con pandas...
🧹 Paso 6: Limpiando datos con pandas...
   - Valores nulos antes:
order_id           0
fecha              0
cliente_id         0
cliente_nombre     0
region             0
producto           1
categoria          0
cantidad           0
precio_unitario    0
descuento          0
vendedor           0
dtype: int64
💾 Paso 7: Guardando archivo final...

📊 RESULTADOS DE LA LIMPIEZA
✅ Registros originales: 60
📋 Columnas: ['order_id', 'fecha', 'cliente_id', 'cliente_nombre', 'region', 'producto', 'categoria', 'cantidad', 'precio_unitario', 'descuento', 'vendedor', 'ingreso']

🔍 Verificando nombres de vendedores:
   Vendedores únicos: ['Ana López', 'Carlos Ruiz', 'Luis Mora', 'Maria Torres']

🔍 Verificando productos con comillas:

🔍 Verificando orden 1003 (problema original):
   ✅ 

## PASO 8 - Crear columnas

In [12]:
# Columna: Ingreso Bruto
df['ingreso_bruto'] = (
    df['cantidad'] *
    df['precio_unitario']
)

# Columna: Ingreso Neto
df['ingreso_neto'] = (
    df['ingreso_bruto'] *
    (1 - df['descuento'])
)

# Columna: Mes
df['mes'] = df['fecha'].dt.to_period('M')

## PASO 9 - Resumen mensual

In [13]:
resumen = df.groupby('mes').agg(
    total=('ingreso_neto', 'sum'),
    pedidos=('order_id', 'count')
).reset_index()  # <--- Convierte el índice en columna

print(resumen)

       mes     total  pedidos
0  2024-01  13773.50       10
1  2024-02  13050.00       10
2  2024-03  14454.00       11
3  2024-04  16689.00       11
4  2024-05  11387.25       11
5  2024-06  11209.25        7


## PASO 10 - Meta mensual

In [14]:
# Crear DataFrame de Metas
metas = pd.DataFrame({
    'mes': ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06'],
    'meta': [30000, 30000, 30000, 30000, 30000, 30000]
})

## PASO 11 - Merge

In [15]:
# PASO 11 - Convertir y Merge
resumen['mes'] = resumen['mes'].astype(str)  # <--- SOLUCIÓN

resumen = resumen.merge(metas, on='mes')

print(resumen)

       mes     total  pedidos   meta
0  2024-01  13773.50       10  30000
1  2024-02  13050.00       10  30000
2  2024-03  14454.00       11  30000
3  2024-04  16689.00       11  30000
4  2024-05  11387.25       11  30000
5  2024-06  11209.25        7  30000


## PASO 12 - % de cumplimiento

In [16]:
# 1. Crear la columna de cumplimiento
resumen['cumplimiento'] = (resumen['total'] / resumen['meta']) * 100

# 2. Función para colores
def color_cumplimiento(valor):
    if valor >= 100:
        return '🟢'  # Verde - meta alcanzada
    elif valor >= 80:
        return '🟡'  # Amarillo - cerca de la meta
    else:
        return '🔴'  # Rojo - lejos de la meta

# 3. Crear columna status
resumen['status'] = resumen['cumplimiento'].apply(color_cumplimiento)

# 4. Imprimir (selección de columnas)
print(resumen[['mes', 'total', 'meta', 'cumplimiento', 'status']])

       mes     total   meta  cumplimiento status
0  2024-01  13773.50  30000     45.911667      🔴
1  2024-02  13050.00  30000     43.500000      🔴
2  2024-03  14454.00  30000     48.180000      🔴
3  2024-04  16689.00  30000     55.630000      🔴
4  2024-05  11387.25  30000     37.957500      🔴
5  2024-06  11209.25  30000     37.364167      🔴


## PASO 13 - Exportar a SQLite

In [17]:
# Crear una conexión a una base de datos en MEMORIA.
import sqlite3

conn = sqlite3.connect(":memory:")

# Convertir columnas de tipo Period a string antes de exportar
if 'mes' in df.columns:
    df['mes'] = df['mes'].astype(str)

# También verificar otras columnas que puedan ser Period
for col in df.columns:
    if pd.api.types.is_period_dtype(df[col]):
        df[col] = df[col].astype(str)

# Exportar a SQLite
df.to_sql(
    'ventas',
    conn,
    if_exists='replace',
    index=False
)

# Verificar
pd.read_sql(
    'SELECT COUNT(*) FROM ventas',
    conn
)

C:\Users\jorge\AppData\Local\Temp\ipykernel_17272\836237386.py:12: Pandas4Warning: is_period_dtype is deprecated and will be removed in a future version. Use `isinstance(dtype, pd.PeriodDtype)` instead
  if pd.api.types.is_period_dtype(df[col]):


,COUNT(*)
0,60
